# 01.4 Dataset and DataLoader

This notebook answers a practical question: how does data reach a model batch by batch during training? A Dataset defines individual samples. A DataLoader turns those samples into batches, optionally shuffles them, and gives the training loop an iterable interface.

If this notebook is unclear, later training loops become difficult to read because every loop starts with data coming from a loader.

## Learning Goals

After this notebook, you should be able to:

1. Understand the different roles of `Dataset` and `DataLoader`.
2. Implement a minimal custom `Dataset`.
3. Use `DataLoader` to create batches.
4. Understand the roles of `batch_size` and `shuffle`.
5. Read the shapes inside a batch.
6. Prepare the data input interface for later training loops.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

## The Difference Between `Dataset` and `DataLoader`

A `Dataset` answers the question "what is one sample?" It defines how many samples exist and how to retrieve sample `i`. A `DataLoader` answers the question "how do I iterate through samples in batches?" It handles batching, optional shuffling, and the repeated iteration pattern used in training.

You can remember the split this way: the Dataset owns the content, and the DataLoader owns the organization and delivery of that content.

## A Quick Start with Built-in `TensorDataset`

Before implementing a custom dataset, we start with `TensorDataset` to feel the minimal workflow.


In [ ]:
X = torch.tensor(
    [
        [1.0, 0.5],
        [2.0, 1.0],
        [3.0, 1.5],
        [4.0, 2.0],
        [5.0, 2.5],
        [6.0, 3.0],
    ],
    dtype=torch.float32,
)
y = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)

dataset = TensorDataset(X, y)

print("number of samples / number of samples:", len(dataset))
print("sample 0 / sample 0:", dataset[0])
print("sample 3 / sample 3:", dataset[3])

In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_idx, (xb, yb) in enumerate(loader):
    print(f"batch {batch_idx}")
    print("xb =\n", xb)
    print("yb =", yb)
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print()

The most important thing in this example is the shape change. One sample has feature shape `(2,)`, but one batch has feature shape `(batch_size, 2)`. The labels also become a batch-shaped tensor with shape `(batch_size,)`. In other words, DataLoader stacks individual samples and adds the batch dimension in front.

In [ ]:
# Exercise 1
#
# Create a DataLoader from the dataset defined above.
#
# Requirements:
# - batch_size should be 3.
# - shuffle should be False so the order stays predictable.
#
# Then iterate through loader_ex and print xb.shape and yb.shape for each batch.
# This lets you see how DataLoader adds a batch dimension in front of each
# sample's feature shape.

# loader_ex =
# for xb, yb in loader_ex:
#     print(xb.shape, yb.shape)

In [ ]:
# Exercise 1 Reference Solution

loader_ex = DataLoader(dataset, batch_size=3, shuffle=False)
for xb, yb in loader_ex:
    print(xb.shape, yb.shape)

## Building a Custom `Dataset`

In real projects, your data may come from files, tables, tokenized text, or other sources. A custom Dataset lets you define how raw data becomes one model-ready sample. The minimal contract is small: `__len__` returns the number of samples, and `__getitem__(index)` returns one sample at a given index.

Once those two methods are correct, DataLoader can take care of batching and shuffling.

In [ ]:
class SimpleTabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

        assert len(self.features) == len(self.labels), (
            "features and labels must have the same length / features and labels must have the same length"
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        x = self.features[index]
        y = self.labels[index]
        return x, y


features = [
    [1.0, 0.2],
    [2.0, 0.4],
    [3.0, 0.7],
    [4.0, 0.9],
]
labels = [0, 0, 1, 1]

custom_ds = SimpleTabularDataset(features, labels)
print("len(custom_ds) =", len(custom_ds))
print("custom_ds[2] =", custom_ds[2])

This custom Dataset does three important jobs. It converts the raw inputs into tensors, checks that features and labels have the same number of samples, and returns one feature-label pair for a requested index. Those responsibilities are exactly what DataLoader relies on when it builds batches.

In [ ]:
# Exercise 2
#
# Implement a Dataset that returns a dictionary for each sample.
#
# Desired output for ds[0]:
# {"features": first_feature_tensor, "label": first_label_tensor}
#
# __len__ responsibility:
# - Return the number of labels, which should match the number of feature rows.
#
# __getitem__(index) responsibility:
# - Read one row from self.features.
# - Read the matching label from self.labels.
# - Return them in a dict with keys "features" and "label".

class DictDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = DictDataset(features, labels)
# print(len(ds))
# print(ds[0])

In [ ]:
# Exercise 2 Reference Solution

class DictDatasetSolution(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return {
            "features": self.features[index],
            "label": self.labels[index],
        }


ds = DictDatasetSolution(features, labels)
print(len(ds))
print(ds[0])

## `batch_size` and `shuffle`

`batch_size` controls how many samples are grouped together before being passed to the model. Larger batches mean fewer steps per epoch, but they use more memory and can change optimization behavior. `shuffle=True` randomizes sample order, which is usually helpful for training because the model should not learn from a fixed ordering pattern.

Validation and test loaders usually use `shuffle=False` so evaluation is deterministic and easier to inspect.

In [ ]:
torch.manual_seed(42)

loader_no_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=False)
loader_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=True)

print("no shuffle / no shuffle")
for xb, yb in loader_no_shuffle:
    print(xb[:, 0], yb)

print()
print("shuffle / shuffle")
for xb, yb in loader_shuffle:
    print(xb[:, 0], yb)

## What a Batch Actually Looks Like

Understanding batch shapes is foundational for reading model inputs and outputs. A single feature vector might have shape `(num_features,)`, while a batch of those feature vectors has shape `(batch_size, num_features)`. A single label might be a scalar, while a batch of labels has shape `(batch_size,)`. The batch dimension is the dimension DataLoader adds by stacking samples together.

In [ ]:
loader = DataLoader(custom_ds, batch_size=3, shuffle=False)
xb, yb = next(iter(loader))

print("xb =\n", xb)
print("yb =", yb)
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

In [ ]:
# Exercise 3
#
# Create a DataLoader from custom_ds.
#
# Requirements:
# - batch_size should be 4.
# - shuffle should be False.
#
# Then take the first batch with next(iter(loader_big)).
# Print:
# - xb.shape
# - yb.shape
# - xb[0], the first feature row in the batch
# - yb[0], the first label in the batch

# loader_big =
# xb, yb = next(iter(loader_big))
# print(xb.shape)
# print(yb.shape)
# print(xb[0])
# print(yb[0])

In [ ]:
# Exercise 3 Reference Solution

loader_big = DataLoader(custom_ds, batch_size=4, shuffle=False)
xb, yb = next(iter(loader_big))
print(xb.shape)
print(yb.shape)
print(xb[0])
print(yb[0])

## Integrated Mini Example

Now we combine `Dataset`, `DataLoader`, and batch iteration into one small example.


In [ ]:
train_loader = DataLoader(custom_ds, batch_size=2, shuffle=True)

for step, (xb, yb) in enumerate(train_loader):
    batch_mean = xb.mean(dim=0)
    print(f"step={step}")
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print("batch mean / batch mean =", batch_mean)
    print()

## Summary

The central distinction is that a Dataset defines samples, while a DataLoader organizes those samples into batches. A minimal Dataset needs `__len__` so PyTorch knows how many samples exist, and `__getitem__` so PyTorch can request one sample by index. `batch_size` changes how many samples are stacked together, and `shuffle=True` changes the order in which training samples are seen.

Before moving on, make sure you can explain why training loaders usually shuffle data while validation and test loaders usually do not.